In [ ]:
import os
import re
import glob
import random
import warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import f1_score, classification_report
from lightgbm import LGBMClassifier

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False

warnings.filterwarnings("ignore")

# =====================================================================
# 0. CONFIGURATION & REPRODUCIBILITY
# =====================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running V3 Kinematic-Cascade Pipeline on: {DEVICE}")

EMG_COLS = [f'ch{i}' for i in range(1, 9)]
IMU_COLS = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
AUX_REG_COLS = [
    'index_curl', 'middle_curl', 'ring_curl', 'pinky_curl',
    'pinch', 'openness', 'thumb_up', 'roll', 'pitch', 'yaw'
]

GESTURES = ['rest', 'fist', 'open', 'point', 'pinch', 'thumb-up']
GESTURE_MAP = {g: i for i, g in enumerate(GESTURES)}
INV_GESTURE_MAP = {i: g for i, g in enumerate(GESTURES)}

# Anatomical kinematic prototypes: [index, middle, ring, pinky, pinch, openness, thumb_up]
# Used to densify missing optical NaN rows modulated by real EMG muscle force!
ANATOMICAL_PROTOTYPES = np.array([
    [0.15, 0.15, 0.15, 0.15, 0.00, 0.50, 0.00],  # 0: rest
    [0.95, 0.95, 0.95, 0.95, 0.10, 0.00, 0.00],  # 1: fist
    [0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 0.10],  # 2: open
    [0.05, 0.90, 0.90, 0.90, 0.00, 0.25, 0.05],  # 3: point
    [0.75, 0.20, 0.20, 0.20, 0.95, 0.45, 0.00],  # 4: pinch
    [0.90, 0.90, 0.90, 0.90, 0.00, 0.15, 1.00],  # 5: thumb-up
], dtype=np.float32)

SEQ_LEN = 96
WINDOW_SEC = 1.4
TRAIN_STRIDE_SEC = 0.45
BATCH_SIZE = 64
EPOCHS = 42
LR = 1.2e-3
N_FOLDS = 5
CACHE_PATH = "/kaggle/working/aura_v3_kinematic_cache.npz"

DATA_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if any(f.startswith('train_') and f.endswith('.xlsx') for f in files):
        DATA_DIR = os.path.dirname(root) if os.path.basename(root) == 'train' else root
        break
if DATA_DIR is None:
    DATA_DIR = '/kaggle/input/competitions/vr-calibration-free-emg-challenge-pieeg'

def natural_key(path):
    nums = re.findall(r'\d+', os.path.basename(path))
    return int(nums[0]) if nums else 0

def read_aura_excel_fast(fpath):
    xl = pd.ExcelFile(fpath, engine='openpyxl')
    sheet = 'multimodal' if 'multimodal' in xl.sheet_names else xl.sheet_names[0]
    df = xl.parse(sheet)
    df.columns = [str(c).replace('#', '').strip() for c in df.columns]
    return df

# =====================================================================
# 1. BIOMECHANICAL ENVELOPE & 3D POSE RECONSTRUCTION
# =====================================================================
def extract_window_representations(df_win, seq_len=SEQ_LEN):
    """
    1. Isolates true hardware updates (emg_new==1, imu_new==1).
    2. Strips slow baseline drift using local rolling median/mean removal.
    3. Rectifies EMG into Slow Force Envelopes, Fast Firing Envelopes, and Spatial Ratios
       BEFORE resampling so positive/negative AC spikes never cancel out!
    4. Integrates IMU angular velocity (gx, gy, gz) to reconstruct 3D wrist angles (roll, pitch, yaw).
    """
    # --- A. Clean Native-Resolution EMG ---
    if 'emg_new' in df_win.columns:
        emg_mask = df_win['emg_new'].values == 1
        emg_raw = df_win.loc[emg_mask, EMG_COLS].astype(np.float32).values if np.sum(emg_mask) >= 12 else df_win[EMG_COLS].astype(np.float32).values
    else:
        emg_raw = df_win[EMG_COLS].astype(np.float32).values

    emg_df = pd.DataFrame(emg_raw).interpolate(limit_direction='both').fillna(0.0)
    # Remove both window mean AND local 25-sample slow drift (eliminates skin impedance drift)
    emg_centered = emg_df - emg_df.mean(axis=0)
    emg_clean = (emg_centered - emg_centered.rolling(window=25, min_periods=1, center=True).mean()).values

    # First-order motor-unit spike derivative at native resolution
    emg_diff = np.diff(emg_clean, axis=0, prepend=emg_clean[:1, :])

    # Rectified Muscle Force Envelopes (prevents AC phase cancellation during resampling!)
    env_slow = np.sqrt(pd.DataFrame(emg_clean ** 2).rolling(window=15, min_periods=1, center=True).mean().values + 1e-6)
    env_fast = np.sqrt(pd.DataFrame(emg_diff ** 2).rolling(window=7, min_periods=1, center=True).mean().values + 1e-6)

    # Calibration-Free Spatial Muscle Activation Map (sum across 8 channels = 1.0 at each step)
    spatial_map = env_slow / (np.sum(env_slow, axis=1, keepdims=True) + 1e-4)

    # --- B. Clean Native-Resolution IMU & Integrated 3D Wrist Pose ---
    if 'imu_new' in df_win.columns:
        imu_mask = df_win['imu_new'].values == 1
        imu_raw = df_win.loc[imu_mask, IMU_COLS].astype(np.float32).values if np.sum(imu_mask) >= 4 else df_win[IMU_COLS].astype(np.float32).values
    else:
        imu_raw = df_win[IMU_COLS].astype(np.float32).values

    imu_df = pd.DataFrame(imu_raw).interpolate(limit_direction='both').fillna(0.0)
    imu_clean = (imu_df - imu_df.mean(axis=0)).values

    # Resample EMG streams (32 ch) & IMU stream (6 ch) to uniform grid (SEQ_LEN = 96)
    grid = np.linspace(0.0, 1.0, seq_len)
    emg_x = np.linspace(0.0, 1.0, len(emg_clean))
    imu_x = np.linspace(0.0, 1.0, len(imu_clean))

    emg_osc_g = np.zeros((seq_len, 8), dtype=np.float32)
    env_slow_g = np.zeros((seq_len, 8), dtype=np.float32)
    env_fast_g = np.zeros((seq_len, 8), dtype=np.float32)
    spatial_g = np.zeros((seq_len, 8), dtype=np.float32)
    for c in range(8):
        emg_osc_g[:, c] = np.interp(grid, emg_x, np.sign(emg_clean[:, c]) * np.log1p(np.abs(emg_clean[:, c])))
        env_slow_g[:, c] = np.interp(grid, emg_x, np.log1p(env_slow[:, c]))
        env_fast_g[:, c] = np.interp(grid, emg_x, np.log1p(env_fast[:, c]))
        spatial_g[:, c] = np.interp(grid, emg_x, spatial_map[:, c])

    imu_vel_g = np.zeros((seq_len, 6), dtype=np.float32)
    for c in range(6):
        imu_vel_g[:, c] = np.interp(grid, imu_x, imu_clean[:, c])

    # Cumulative integral of IMU reconstructs continuous 3D wrist angle (roll, pitch, yaw) & position!
    imu_pose_g = np.cumsum(imu_vel_g, axis=0) / float(seq_len)
    imu_Pose_mean_centered = imu_pose_g - np.mean(imu_pose_g, axis=0, keepdims=True)
    imu_acc_g = np.diff(imu_vel_g, axis=0, prepend=imu_vel_g[:1, :])

    # Stack 50 Biomechanical Sequence Channels: 8+8+8+8 + 6+6+6 = 50 channels
    seq_tensor = np.concatenate([
        emg_osc_g, env_slow_g, env_fast_g, spatial_g,
        imu_vel_g, imu_Pose_mean_centered, imu_acc_g
    ], axis=1).T.astype(np.float32)  # Shape: (50, 96)

    # --- C. Extract Rich Biomechanical & Riemannian Tabular Features ---
    feats = []
    rms_c = np.sqrt(np.mean(emg_clean ** 2, axis=0)) + 1e-5
    mav_c = np.mean(np.abs(emg_clean), axis=0) + 1e-5
    wl_c = np.sum(np.abs(emg_diff), axis=0) / float(len(emg_clean)) + 1e-5
    p90_c = np.percentile(np.abs(emg_clean), 90, axis=0) + 1e-5
    zc_c = np.sum(np.diff(np.sign(emg_clean), axis=0) != 0, axis=0) / float(len(emg_clean))

    # 1. Log-Energy & Complexity (40 features)
    feats.extend(np.log1p(rms_c))
    feats.extend(np.log1p(mav_c))
    feats.extend(np.log1p(wl_c))
    feats.extend(np.log1p(p90_c))
    feats.extend(zc_c)

    # 2. Calibration-Free Spatial Distribution (24 features)
    rms_ratio = rms_c / np.sum(rms_c)
    wl_ratio = wl_c / np.sum(wl_c)
    p90_ratio = p90_c / np.sum(p90_c)
    feats.extend(rms_ratio)
    feats.extend(wl_ratio)
    feats.extend(p90_ratio)

    # Global contractile power & spatial entropy (6 features)
    total_power = np.log1p(np.sum(rms_c))
    total_wl = np.log1p(np.sum(wl_c))
    entropy = -np.sum(rms_ratio * np.log(rms_ratio + 1e-6))
    max_to_mean = np.max(rms_ratio) / (np.mean(rms_ratio) + 1e-6)
    feats.extend([total_power, total_wl, entropy, max_to_mean, float(np.argmax(rms_ratio)), float(np.argmin(rms_ratio))])

    # 3. Opposing & Adjacent Forearm Muscle Antagonism (16 features)
    for i in range(8):
        opp = (i + 4) % 8
        adj = (i + 1) % 8
        feats.append(float(rms_ratio[i] - rms_ratio[opp]))
        feats.append(float(rms_ratio[i] / (rms_ratio[adj] + 1e-4)))

    # 4. 3-Phase Temporal Evolution: Onset vs Hold vs Release (30 features)
    for sub in np.array_split(env_slow_g, 3, axis=0):
        sub_mean = np.mean(sub, axis=0)
        feats.extend(sub_mean)
        feats.extend([float(np.sum(sub_mean)), float(np.std(sub_mean))])
    # Hold-to-Onset Ratio
    onset_m = np.mean(env_slow_g[:32], axis=0) + 1e-4
    hold_m = np.mean(env_slow_g[32:64], axis=0) + 1e-4
    feats.extend(hold_m - onset_m)

    # 5. Riemannian Tangential Spatial Covariance Matrix (36 features)
    # Captures exact cross-electrode muscle co-activation on the SPD manifold!
    norm_e = emg_clean / (rms_c[None, :] + 1e-4)
    cov_mat = (norm_e.T @ norm_e) / float(len(norm_e)) + 1e-3 * np.eye(8)
    eigvals, eigvecs = np.linalg.eigh(cov_mat)
    eigvals = np.clip(eigvals, 1e-4, None)
    log_cov = eigvecs @ np.diag(np.log(eigvals)) @ eigvecs.T
    iu8 = np.triu_indices(8, k=0)
    feats.extend(log_cov[iu8])
    feats.extend(np.log(eigvals))  # Sorted spatial eigenvalues!

    # 6. Frequency Spectral Power Ratios (24 features)
    fft_p = np.abs(np.fft.rfft(emg_clean, axis=0)) ** 2
    if len(fft_p) > 6:
        b1, b2, b3 = np.array_split(fft_p[1:], 3, axis=0)
        p1 = np.mean(b1, axis=0) + 1e-5
        p2 = np.mean(b2, axis=0) + 1e-5
        p3 = np.mean(b3, axis=0) + 1e-5
        ptot = p1 + p2 + p3
        feats.extend(p1 / ptot)
        feats.extend(p2 / ptot)
        feats.extend(p3 / ptot)
    else:
        feats.extend(np.zeros(24))

    # 7. Full 3D IMU Kinematics & Reconstructed Orientation (48 features)
    imu_rms = np.sqrt(np.mean(imu_vel_g ** 2, axis=0))
    imu_max = np.max(np.abs(imu_vel_g), axis=0)
    imu_disp = imu_pose_g[-1, :] - imu_pose_g[0, :]       # Net 3D angle & position change!
    imu_pose_std = np.std(imu_Pose_mean_centered, axis=0)
    imu_mid_minus_start = np.mean(imu_vel_g[32:64], axis=0) - np.mean(imu_vel_g[:32], axis=0)
    imu_end_minus_mid = np.mean(imu_vel_g[64:], axis=0) - np.mean(imu_vel_g[32:64], axis=0)

    feats.extend(np.log1p(imu_rms))
    feats.extend(np.log1p(imu_max))
    feats.extend(imu_disp)
    feats.extend(imu_pose_std)
    feats.extend(imu_mid_minus_start)
    feats.extend(imu_end_minus_mid)

    acc_mag = np.sqrt(np.sum(imu_vel_g[:, :3] ** 2, axis=1))
    gyro_mag = np.sqrt(np.sum(imu_vel_g[:, 3:] ** 2, axis=1))
    feats.extend([
        float(np.log1p(np.mean(acc_mag))), float(np.log1p(np.std(acc_mag))),
        float(np.log1p(np.mean(gyro_mag))), float(np.log1p(np.std(gyro_mag)))
    ])

    imu_corr = np.nan_to_num(np.corrcoef(imu_vel_g.T), nan=0.0)
    feats.extend(imu_corr[np.triu_indices(6, k=1)])

    tab_vec = np.nan_to_num(np.array(feats, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return seq_tensor, tab_vec

# =====================================================================
# 2. SESSION-AWARE CALIBRATION (ELIMINATES CROSS-PERSON IMPEDANCE SHIFT)
# =====================================================================
def append_session_calibrated_features(X_seq, X_tab, session_ids):
    """
    Calibrates every window against its own recording session (train_k.xlsx or test_k.xlsx).
    Immediately distinguishes 'rest' (bottom of user's session energy) from 'fist'/'open'
    (top of user's session energy), solving cross-user forearm variance!
    """
    n_samples = len(X_tab)
    # First 46 columns of X_tab are channel log-energies & global contractile power
    energy_cols = X_tab[:, :46].copy()
    sess_z_feats = np.zeros_like(energy_cols)
    sess_pct_feats = np.zeros((n_samples, 10), dtype=np.float32)

    X_seq_cal = X_seq.copy()

    for s_id in np.unique(session_ids):
        mask = (session_ids == s_id)
        if np.sum(mask) < 2:
            continue

        # 1. Calibrate Tabular Energy Features per Session
        sub_e = energy_cols[mask]
        med = np.median(sub_e, axis=0, keepdims=True)
        iqr = (np.percentile(sub_e, 75, axis=0, keepdims=True) - np.percentile(sub_e, 25, axis=0, keepdims=True)) + 1e-4
        sess_z_feats[mask] = np.clip((sub_e - med) / iqr, -6.0, 6.0)

        # Rank-percentile of total EMG power (col 40) & 8 channel powers (cols 0..7) + gyro power within session
        key_cols = [0, 1, 2, 3, 4, 5, 6, 7, 40, 41]
        for idx_k, col_k in enumerate(key_cols):
            vals = X_tab[mask, col_k]
            order = np.argsort(np.argsort(vals))
            sess_pct_feats[mask, idx_k] = order / float(max(1, len(vals) - 1))

        # 2. Calibrate Deep Sequence Envelopes (channels 8..24) per Session
        sub_seq = X_seq[mask, 8:24, :]
        s_med = np.median(sub_seq, axis=(0, 2), keepdims=True)
        s_iqr = (np.percentile(sub_seq, 75, axis=(0, 2), keepdims=True) - np.percentile(sub_seq, 25, axis=(0, 2), keepdims=True)) + 1e-4
        X_seq_cal[mask, 8:24, :] = np.clip((sub_seq - s_med) / s_iqr, -6.0, 6.0)

    X_tab_augmented = np.concatenate([X_tab, sess_z_feats, sess_pct_feats], axis=1)
    return X_seq_cal, X_tab_augmented

# =====================================================================
# 3. PARALLEL DATA LOADER WITH DENSE KINEMATIC TARGET SYNTHESIS
# =====================================================================
def process_single_train_file(args):
    session_idx, fpath = args
    df = read_aura_excel_fast(fpath)
    if 'label_source' in df.columns:
        df = df[df['label_source'].astype(str).str.strip().str.lower() == 'guided'].reset_index(drop=True)

    label_col = 'label' if 'label' in df.columns else 'gesture'
    df[label_col] = df[label_col].astype(str).str.strip().str.lower()
    df = df[df[label_col].isin(GESTURE_MAP.keys())].reset_index(drop=True)
    if len(df) == 0:
        return None

    time_jump = df['t'].diff().abs() > 0.25
    label_change = df[label_col] != df[label_col].shift()
    df['seg_id'] = (time_jump | label_change).cumsum()

    seq_list, tab_list, cls_list, reg_list, mask_list, is_anchor_list = [], [], [], [], [], []

    for _, seg in df.groupby('seg_id'):
        t_start, t_end = seg['t'].iloc[0], seg['t'].iloc[-1]
        dur = t_end - t_start
        gesture_id = GESTURE_MAP[seg[label_col].iloc[0]]

        # Skip initial 0.12s human visual reaction delay if segment is long enough
        offset = 0.12 if dur >= 1.65 else 0.0
        curr_t = t_start + offset
        win_idx = 0

        while curr_t + WINDOW_SEC <= t_end + 0.08:
            win = seg[(seg['t'] >= curr_t) & (seg['t'] < curr_t + WINDOW_SEC)]
            if len(win) >= 32:
                seq_t, tab_v = extract_window_representations(win, seq_len=SEQ_LEN)

                reg_t = np.zeros(10, dtype=np.float32)
                reg_m = np.zeros(10, dtype=np.float32)
                for idx, col in enumerate(AUX_REG_COLS):
                    if col in win.columns:
                        vals = pd.to_numeric(win[col], errors='coerce').dropna().values
                        if len(vals) > 0:
                            reg_t[idx] = float(np.mean(vals))
                            reg_m[idx] = 1.0

                seq_list.append(seq_t)
                tab_list.append(tab_v)
                cls_list.append(gesture_id)
                reg_list.append(reg_t)
                mask_list.append(reg_m)
                # Mark non-overlapping windows (every 3rd stride) for leak-free validation
                is_anchor_list.append(1 if (win_idx % 3 == 0) else 0)
                win_idx += 1
            curr_t += TRAIN_STRIDE_SEC

    if not seq_list:
        return None
    n = len(seq_list)
    return (
        np.stack(seq_list), np.stack(tab_list), np.array(cls_list, dtype=np.int64),
        np.stack(reg_list), np.stack(mask_list),
        np.full(n, session_idx, dtype=np.int64), np.array(is_anchor_list, dtype=np.int64)
    )

def load_or_build_all_data(data_dir):
    if os.path.exists(CACHE_PATH):
        print(f"Loading cached processed dataset from {CACHE_PATH}...")
        data = np.load(CACHE_PATH, allow_pickle=True)
        return (
            data['X_seq'], data['X_tab'], data['y_cls'], data['y_kin_dense'],
            data['kin_weights'], data['groups'], data['is_anchor'],
            data['X_test_seq'], data['X_test_tab'], data['test_ids'].tolist()
        )

    train_files = sorted(glob.glob(os.path.join(data_dir, "**", "train_*.xlsx"), recursive=True), key=natural_key)
    test_files = sorted(glob.glob(os.path.join(data_dir, "**", "test_*.xlsx"), recursive=True), key=natural_key)
    print(f"Parallel parsing {len(train_files)} train files and {len(test_files)} test files...")

    with ThreadPoolExecutor(max_workers=4) as ex:
        results = list(ex.map(process_single_train_file, enumerate(train_files)))

    results = [r for r in results if r is not None]
    X_seq = np.concatenate([r[0] for r in results], axis=0)
    X_tab = np.concatenate([r[1] for r in results], axis=0)
    y_cls = np.concatenate([r[2] for r in results], axis=0)
    y_reg_raw = np.concatenate([r[3] for r in results], axis=0)
    reg_mask = np.concatenate([r[4] for r in results], axis=0)
    groups = np.concatenate([r[5] for r in results], axis=0)
    is_anchor = np.concatenate([r[6] for r in results], axis=0)

    # Parse Test Set (preserving exact file & action_id order)
    X_test_seq, X_test_tab, test_ids, test_sess = [], [], [], []
    for s_idx, fpath in enumerate(test_files):
        df = read_aura_excel_fast(fpath)
        id_col = 'action_id' if 'action_id' in df.columns else df.columns[0]
        for act_id, win in df.groupby(id_col, sort=False):
            s_t, t_v = extract_window_representations(win, seq_len=SEQ_LEN)
            X_test_seq.append(s_t)
            X_test_tab.append(t_v)
            test_ids.append(act_id)
            test_sess.append(s_idx)

    X_test_seq = np.stack(X_test_seq)
    X_test_tab = np.stack(X_test_tab)
    test_sess = np.array(test_sess, dtype=np.int64)

    # Apply Session-Aware Calibration to both Train and Test!
    X_seq, X_tab = append_session_calibrated_features(X_seq, X_tab, groups)
    X_test_seq, X_test_tab = append_session_calibrated_features(X_test_seq, X_test_tab, test_sess)

    # Global Standardization of Sequence Channels
    ch_mu = np.mean(X_seq, axis=(0, 2), keepdims=True)
    ch_sd = np.std(X_seq, axis=(0, 2), keepdims=True) + 1e-5
    X_seq = np.clip((X_seq - ch_mu) / ch_sd, -8.0, 8.0)
    X_test_seq = np.clip((X_test_seq - ch_mu) / ch_sd, -8.0, 8.0)

    # Construct 100% DENSE Biomechanical Kinematic Targets (Solves NaN starvation!)
    # Standardize observed optical/IMU targets where present
    for j in range(10):
        valid = reg_mask[:, j] > 0.5
        if np.sum(valid) > 10:
            mu_j = np.mean(y_reg_raw[valid, j])
            sd_j = np.std(y_reg_raw[valid, j]) + 1e-5
            y_reg_raw[valid, j] = np.clip((y_reg_raw[valid, j] - mu_j) / sd_j, -4.0, 4.0)

    # Synthesize dense targets for missing cells using anatomical prototypes + real IMU/EMG energy
    proto_norm = (ANATOMICAL_PROTOTYPES - np.mean(ANATOMICAL_PROTOTYPES, axis=0, keepdims=True)) / (np.std(ANATOMICAL_PROTOTYPES, axis=0, keepdims=True) + 1e-5)
    y_kin_dense = np.zeros((len(X_seq), 10), dtype=np.float32)
    kin_weights = np.ones((len(X_seq), 10), dtype=np.float32)

    # Extract normalized IMU net displacement (cols 192:195 in X_tab) for roll/pitch/yaw proxy
    imu_disp_proxy = X_tab[:, 182:185].copy()
    imu_disp_proxy = (imu_disp_proxy - np.mean(imu_disp_proxy, axis=0, keepdims=True)) / (np.std(imu_disp_proxy, axis=0, keepdims=True) + 1e-5)

    for i in range(len(X_seq)):
        g = y_cls[i]
        # Finger curls & openness (0..6): blend observed optical value or anatomical prototype
        for j in range(7):
            if reg_mask[i, j] > 0.5:
                y_kin_dense[i, j] = y_reg_raw[i, j]
                kin_weights[i, j] = 2.0  # Higher weight on real optical camera measurements
            else:
                y_kin_dense[i, j] = proto_norm[g, j]
                kin_weights[i, j] = 1.0
        # Orientation roll/pitch/yaw (7..9): use observed or integrated IMU displacement proxy
        for j in range(7, 10):
            if reg_mask[i, j] > 0.5:
                y_kin_dense[i, j] = y_reg_raw[i, j]
                kin_weights[i, j] = 2.0
            else:
                y_kin_dense[i, j] = np.clip(imu_disp_proxy[i, j - 7], -3.0, 3.0)
                kin_weights[i, j] = 0.8

    np.savez_compressed(
        CACHE_PATH, X_seq=X_seq, X_tab=X_tab, y_cls=y_cls, y_kin_dense=y_kin_dense,
        kin_weights=kin_weights, groups=groups, is_anchor=is_anchor,
        X_test_seq=X_test_seq, X_test_tab=X_test_tab, test_ids=np.array(test_ids, dtype=object)
    )
    print(f"Cached dataset to {CACHE_PATH} | Train: {X_seq.shape}, Tabular: {X_tab.shape}, Test: {X_test_seq.shape}")
    return X_seq, X_tab, y_cls, y_kin_dense, kin_weights, groups, is_anchor, X_test_seq, X_test_tab, test_ids

# =====================================================================
# 4. TIGHTLY-COUPLED KINEMATIC CASCADE NETWORK (CNN <-> REGRESSION <-> CLS)
# =====================================================================
class AuraCascadeDataset(Dataset):
    def __init__(self, X_seq, X_tab, y_cls=None, y_kin=None, kin_w=None, is_train=True):
        self.X_seq = X_seq
        self.X_tab = X_tab
        self.y_cls = y_cls
        self.y_kin = y_kin
        self.kin_w = kin_w
        self.is_train = is_train

    def __len__(self):
        return len(self.X_seq)

    def __getitem__(self, idx):
        seq = self.X_seq[idx].copy()
        tab = self.X_tab[idx].copy()

        if self.is_train:
            if random.random() < 0.55:
                shift = random.randint(-6, 6)
                seq = np.roll(seq, shift=shift, axis=-1)
            if random.random() < 0.40:
                scale = np.random.uniform(0.90, 1.10, size=(seq.shape[0], 1)).astype(np.float32)
                noise = np.random.normal(0.0, 0.02, size=seq.shape).astype(np.float32)
                seq = seq * scale + noise

        if self.y_cls is None:
            return torch.tensor(seq, dtype=torch.float32), torch.tensor(tab, dtype=torch.float32)

        return (
            torch.tensor(seq, dtype=torch.float32),
            torch.tensor(tab, dtype=torch.float32),
            torch.tensor(self.y_cls[idx], dtype=torch.long),
            torch.tensor(self.y_kin[idx], dtype=torch.float32),
            torch.tensor(self.kin_w[idx], dtype=torch.float32)
        )

class DilatedSEBlock1D(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.b1 = nn.Conv1d(in_c, out_c // 2, kernel_size=3, stride=stride, padding=1)
        self.b2 = nn.Conv1d(in_c, out_c // 2, kernel_size=5, stride=stride, padding=4, dilation=2)
        self.bn1 = nn.BatchNorm1d(out_c)
        self.conv2 = nn.Conv1d(out_c, out_c, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(out_c)
        self.se = nn.Sequential(
            nn.Linear(out_c, out_c // 4),
            nn.GELU(),
            nn.Linear(out_c // 4, out_c),
            nn.Sigmoid()
        )
        self.short = nn.Conv1d(in_c, out_c, 1, stride=stride) if (in_c != out_c or stride != 1) else nn.Identity()

    def forward(self, x):
        h = F.gelu(self.bn1(torch.cat([self.b1(x), self.b2(x)], dim=1)))
        h = self.bn2(self.conv2(h))
        w = self.se(h.mean(dim=-1)).unsqueeze(-1)
        return F.gelu(h * w + self.short(x))

class AuraKinematicCascadeNet(nn.Module):
    """
    True Cascaded Architecture where CNN Regression directly drives Gesture Classification:
      Stage 1: Multi-Scale Dilated 1D-CNN extracts temporal muscle/motion features.
      Stage 2: Dense Temporal Kinematic Regressor predicts the 10 finger/wrist curves K(t) at every step t.
      Stage 3: Predicted K(t) is fed BACK into the BiGRU + Self-Attention sequence encoder.
      Stage 4: Regressed window kinematics k_win gate the classifier via FiLM AND project directly to logits!
    """
    def __init__(self, in_channels=50, tab_dim=300, num_classes=6, num_kin=10):
        super().__init__()
        # Stage 1: Multi-Scale CNN Feature Extractor
        self.cnn_stem = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.GELU()
        )
        self.block1 = DilatedSEBlock1D(64, 96, stride=2)   # 96 -> 48 steps
        self.block2 = DilatedSEBlock1D(96, 128, stride=1)  # 48 steps

        # Stage 2: Dense Temporal Kinematic Regression Bottleneck (Predicts 10 kinematics at each of 48 steps)
        self.temporal_kin_head = nn.Sequential(
            nn.Conv1d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Conv1d(64, num_kin, kernel_size=1)          # Output: K(t) shape (B, 10, 48)
        )
        # Bridge that projects predicted K(t) back into the sequence stream
        self.kin_feedback_proj = nn.Sequential(
            nn.Conv1d(num_kin, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.GELU()
        )

        # Stage 3: Sequence Encoder over [CNN_Feats (128) + Kin_Proj (32) + Raw_Kin_Preds (10) = 170]
        self.bigru = nn.GRU(170, 96, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.temporal_attn = nn.Sequential(
            nn.Linear(192, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        # Tabular Biomechanical & Riemannian Branch
        self.tab_encoder = nn.Sequential(
            nn.BatchNorm1d(tab_dim),
            nn.Linear(tab_dim, 160),
            nn.BatchNorm1d(160),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(160, 96),
            nn.BatchNorm1d(96),
            nn.GELU()
        )

        # Refine Window-Level Kinematic Regression from both Sequence K(t) and Tabular Biomechanics
        self.win_kin_refiner = nn.Sequential(
            nn.Linear(num_kin + 192 + 96, 64),
            nn.GELU(),
            nn.Linear(64, num_kin)
        )

        # Stage 4: FiLM Kinematic Gating & Direct Kinematic-to-Gesture Classifier
        # Total fused dimension = 192 (Attn) + 192 (Max) + 96 (Tab) = 480
        self.film_gamma = nn.Linear(num_kin, 480)
        self.film_beta = nn.Linear(num_kin, 480)

        self.cls_mlp = nn.Sequential(
            nn.LayerNorm(480 + num_kin),
            nn.Dropout(0.30),
            nn.Linear(480 + num_kin, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, num_classes)
        )
        # Direct linear skip connection from regressed finger curls -> gesture logits!
        self.kin_to_cls_direct = nn.Linear(num_kin, num_classes)

    def forward(self, x_seq, x_tab):
        h = self.cnn_stem(x_seq)
        h = self.block2(self.block1(h))                     # (B, 128, 48)

        # Predict continuous finger/wrist kinematics K(t) across time
        kin_seq = self.temporal_kin_head(h)                 # (B, 10, 48)
        kin_fb = self.kin_feedback_proj(kin_seq)            # (B, 32, 48)

        # Inject predicted kinematics back into the sequence backbone!
        fused_seq_in = torch.cat([h, kin_fb, kin_seq], dim=1).permute(0, 2, 1)  # (B, 48, 170)
        seq_out, _ = self.bigru(fused_seq_in)               # (B, 48, 192)

        attn_w = F.softmax(self.temporal_attn(seq_out), dim=1)  # (B, 48, 1)
        attn_pool = torch.sum(seq_out * attn_w, dim=1)          # (B, 192)
        max_pool = torch.max(seq_out, dim=1)[0]                 # (B, 192)

        # Attention-weighted temporal average of K(t) + tabular refinement
        kin_temporal_pool = torch.sum(kin_seq.permute(0, 2, 1) * attn_w, dim=1)  # (B, 10)
        tab_emb = self.tab_encoder(x_tab)                                        # (B, 96)

        kin_pred = kin_temporal_pool + 0.3 * self.win_kin_refiner(
            torch.cat([kin_temporal_pool, attn_pool, tab_emb], dim=-1)
        )  # Final Regressed Kinematics: (B, 10)

        # FiLM Modulation: Regressed finger curls scale and shift the deep representation!
        base_feat = torch.cat([attn_pool, max_pool, tab_emb], dim=-1)            # (B, 480)
        gamma = torch.sigmoid(self.film_gamma(kin_pred)) * 2.0
        beta = self.film_beta(kin_pred)
        modulated_feat = base_feat * gamma + beta

        # Final Gesture Logits = Deep Modulated MLP + Direct Kinematic Readout
        logits = self.cls_mlp(torch.cat([modulated_feat, kin_pred], dim=-1)) + 0.5 * self.kin_to_cls_direct(kin_pred)
        return logits, kin_pred, kin_seq

class CoupledCascadeLoss(nn.Module):
    """
    Jointly optimizes:
      1. Gesture Classification (Label-Smoothed Cross-Entropy + Soft Macro-F1)
      2. Window-Level Kinematic Regression (Weighted Smooth-L1)
      3. Temporal Kinematic Trajectory Consistency (Smoothness + Target Alignment)
    """
    def __init__(self, num_classes=6, smoothing=0.05, f1_w=0.35, reg_w=0.45):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=smoothing)
        self.num_classes = num_classes
        self.f1_w = f1_w
        self.reg_w = reg_w

    def forward(self, logits, kin_pred, kin_seq, y_cls, y_kin, kin_w):
        # 1. Classification Loss
        ce_loss = self.ce(logits, y_cls)
        probs = F.softmax(logits, dim=-1)
        one_hot = F.one_hot(y_cls, num_classes=self.num_classes).float()
        tp = torch.sum(probs * one_hot, dim=0)
        fp = torch.sum(probs * (1.0 - one_hot), dim=0)
        fn = torch.sum((1.0 - probs) * one_hot, dim=0)
        soft_f1 = (2.0 * tp + 1e-5) / (2.0 * tp + fp + fn + 1e-5)
        f1_loss = 1.0 - torch.mean(soft_f1)
        cls_loss = (1.0 - self.f1_w) * ce_loss + self.f1_w * f1_loss

        # 2. Dense Window-Level Kinematic Regression Loss
        reg_diff = F.smooth_l1_loss(kin_pred, y_kin, reduction='none')
        win_reg_loss = torch.mean(reg_diff * kin_w)

        # 3. Temporal Trajectory Supervision & Smoothness on K(t)
        seq_target = y_kin.unsqueeze(-1).expand_as(kin_seq)
        seq_reg_loss = torch.mean(F.smooth_l1_loss(kin_seq, seq_target, reduction='none') * kin_w.unsqueeze(-1))
        temporal_smoothness = torch.mean((kin_seq[:, :, 1:] - kin_seq[:, :, :-1]) ** 2)

        return cls_loss + self.reg_w * (win_reg_loss + 0.4 * seq_reg_loss + 0.05 * temporal_smoothness)

# =====================================================================
# 5. TWO-STAGE STACKED TRAINING (CASCADE CNN -> GBDT META-STACKING)
# =====================================================================
(X_seq, X_tab, y_cls, y_kin, kin_w, groups, is_anchor,
 X_test_seq, X_test_tab, test_ids) = load_or_build_all_data(DATA_DIR)

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS)

oof_nn_probs = np.zeros((len(X_seq), 6), dtype=np.float32)
oof_nn_kin = np.zeros((len(X_seq), 10), dtype=np.float32)
test_nn_probs = np.zeros((len(X_test_seq), 6), dtype=np.float32)
test_nn_kin = np.zeros((len(X_test_seq), 10), dtype=np.float32)

oof_lgb_probs = np.zeros((len(X_seq), 6), dtype=np.float32)
oof_cat_probs = np.zeros((len(X_seq), 6), dtype=np.float32)
test_lgb_probs = np.zeros((len(X_test_seq), 6), dtype=np.float32)
test_cat_probs = np.zeros((len(X_test_seq), 6), dtype=np.float32)

criterion = CoupledCascadeLoss()

for fold, (trn_idx, val_idx) in enumerate(sgkf.split(X_seq, y_cls, groups)):
    print(f"\n==================== FOLD {fold + 1} / {N_FOLDS} ====================")
    scaler = RobustScaler()
    X_tab_trn = np.clip(scaler.fit_transform(X_tab[trn_idx]), -10.0, 10.0)
    X_tab_val = np.clip(scaler.transform(X_tab[val_idx]), -10.0, 10.0)
    X_tab_tst = np.clip(scaler.transform(X_test_tab), -10.0, 10.0)

    # --- STAGE 1: Train Tightly-Coupled AuraKinematicCascadeNet ---
    trn_ds = AuraCascadeDataset(X_seq[trn_idx], X_tab_trn, y_cls[trn_idx], y_kin[trn_idx], kin_w[trn_idx], is_train=True)
    val_ds = AuraCascadeDataset(X_seq[val_idx], X_tab_val, y_cls[val_idx], y_kin[val_idx], kin_w[val_idx], is_train=False)
    tst_ds = AuraCascadeDataset(X_test_seq, X_tab_tst, is_train=False)

    trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    tst_loader = DataLoader(tst_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = AuraKinematicCascadeNet(in_channels=X_seq.shape[1], tab_dim=X_tab.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1.5e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

    best_nn_f1 = 0.0
    best_val_p, best_val_k = None, None
    ckpt_path = f"aura_cascade_fold{fold}.pt"

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for bx_s, bx_t, by_c, by_k, bw_k in trn_loader:
            bx_s, bx_t = bx_s.to(DEVICE), bx_t.to(DEVICE)
            by_c, by_k, bw_k = by_c.to(DEVICE), by_k.to(DEVICE), bw_k.to(DEVICE)

            if random.random() < 0.35:
                lam = np.random.beta(0.4, 0.4)
                perm = torch.randperm(bx_s.size(0)).to(DEVICE)
                mx_s = lam * bx_s + (1.0 - lam) * bx_s[perm]
                mx_t = lam * bx_t + (1.0 - lam) * bx_t[perm]
                mk_y = lam * by_k + (1.0 - lam) * by_k[perm]
                mk_w = 0.5 * (bw_k + bw_k[perm])

                logits, kin_p, kin_s = model(mx_s, mx_t)
                loss = (
                    lam * criterion(logits, kin_p, kin_s, by_c, mk_y, mk_w) +
                    (1.0 - lam) * criterion(logits, kin_p, kin_s, by_c[perm], mk_y, mk_w)
                )
            else:
                logits, kin_p, kin_s = model(bx_s, bx_t)
                loss = criterion(logits, kin_p, kin_s, by_c, by_k, bw_k)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step()

        # Validation with 3-Way Time-Shift Test-Time Augmentation (TTA)
        model.eval()
        vp_list, vk_list = [], []
        with torch.no_grad():
            for bx_s, bx_t, _, _, _ in val_loader:
                bx_s, bx_t = bx_s.to(DEVICE), bx_t.to(DEVICE)
                l0, k0, _ = model(bx_s, bx_t)
                l1, k1, _ = model(torch.roll(bx_s, shifts=-3, dims=-1), bx_t)
                l2, k2, _ = model(torch.roll(bx_s, shifts=3, dims=-1), bx_t)

                p = 0.50 * F.softmax(l0, dim=-1) + 0.25 * F.softmax(l1, dim=-1) + 0.25 * F.softmax(l2, dim=-1)
                k = 0.50 * k0 + 0.25 * k1 + 0.25 * k2
                vp_list.append(p.cpu().numpy())
                vk_list.append(k.cpu().numpy())

        vp = np.concatenate(vp_list, axis=0)
        vk = np.concatenate(vk_list, axis=0)
        cur_f1 = f1_score(y_cls[val_idx], np.argmax(vp, axis=1), average='macro')

        if cur_f1 > best_nn_f1:
            best_nn_f1 = cur_f1
            best_val_p = vp
            best_val_k = vk
            torch.save(model.state_dict(), ckpt_path)

        if epoch % 10 == 0 or epoch == EPOCHS:
            print(f"  Epoch {epoch:02d}/{EPOCHS:02d} | Val Macro-F1: {cur_f1:.4f} (Best: {best_nn_f1:.4f})")

    oof_nn_probs[val_idx] = best_val_p
    oof_nn_kin[val_idx] = best_val_k

    # Reload Best Checkpoint to Extract Clean Training Kinematics & Test Predictions
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    trn_eval_loader = DataLoader(
        AuraCascadeDataset(X_seq[trn_idx], X_tab_trn, is_train=False),
        batch_size=BATCH_SIZE, shuffle=False
    )
    trn_kin_preds, trn_nn_probs = [], []
    with torch.no_grad():
        for bx_s, bx_t in trn_eval_loader:
            l0, k0, _ = model(bx_s.to(DEVICE), bx_t.to(DEVICE))
            trn_kin_preds.append(k0.cpu().numpy())
            trn_nn_probs.append(F.softmax(l0, dim=-1).cpu().numpy())
    trn_kin_preds = np.concatenate(trn_kin_preds, axis=0)
    trn_nn_probs = np.concatenate(trn_nn_probs, axis=0)

    # Adaptive BatchNorm Calibration on Test Domain (Unseen Users)
    model.train()
    with torch.no_grad():
        for bx_s, bx_t in tst_loader:
            _ = model(bx_s.to(DEVICE), bx_t.to(DEVICE))
    model.eval()

    tp_list, tk_list = [], []
    with torch.no_grad():
        for bx_s, bx_t in tst_loader:
            bx_s, bx_t = bx_s.to(DEVICE), bx_t.to(DEVICE)
            l0, k0, _ = model(bx_s, bx_t)
            l1, k1, _ = model(torch.roll(bx_s, shifts=-3, dims=-1), bx_t)
            l2, k2, _ = model(torch.roll(bx_s, shifts=3, dims=-1), bx_t)

            p = 0.50 * F.softmax(l0, dim=-1) + 0.25 * F.softmax(l1, dim=-1) + 0.25 * F.softmax(l2, dim=-1)
            k = 0.50 * k0 + 0.25 * k1 + 0.25 * k2
            tp_list.append(p.cpu().numpy())
            tk_list.append(k.cpu().numpy())

    fold_tst_p = np.concatenate(tp_list, axis=0)
    fold_tst_k = np.concatenate(tk_list, axis=0)
    test_nn_probs += fold_tst_p / N_FOLDS
    test_nn_kin += fold_tst_k / N_FOLDS

    # --- STAGE 2: Kinematic-Stacked LightGBM & CatBoost ---
    # Feeds [Tabular Biomechanics + Regressed 10-D Finger Curls + Distance to Anatomical Prototypes + NN Probs]
    proto_norm = (ANATOMICAL_PROTOTYPES - np.mean(ANATOMICAL_PROTOTYPES, axis=0, keepdims=True)) / (
        np.std(ANATOMICAL_PROTOTYPES, axis=0, keepdims=True) + 1e-5
    )

    def build_meta_features(tab_m, kin_m, prob_m):
        # Euclidean similarity between CNN-regressed finger curls (cols 0..6) and the 6 gesture prototypes
        proto_dists = np.zeros((len(kin_m), 6), dtype=np.float32)
        for g_idx in range(6):
            proto_dists[:, g_idx] = -np.sqrt(np.mean((kin_m[:, :7] - proto_norm[g_idx : g_idx + 1, :]) ** 2, axis=1))
        return np.hstack([tab_m, kin_m, proto_dists, prob_m])

    X_stack_trn = build_meta_features(X_tab_trn, trn_kin_preds, trn_nn_probs)
    X_stack_val = build_meta_features(X_tab_val, best_val_k, best_val_p)
    X_stack_tst = build_meta_features(X_tab_tst, fold_tst_k, fold_tst_p)

    lgb_model = LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.025,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.65,
        min_child_samples=20,
        reg_alpha=1.5,
        reg_lambda=2.5,
        random_state=42 + fold,
        n_jobs=-1,
        verbose=-1
    )
    lgb_model.fit(
        X_stack_trn, y_cls[trn_idx],
        eval_set=[(X_stack_val, y_cls[val_idx])]
    )
    oof_lgb_probs[val_idx] = lgb_model.predict_proba(X_stack_val)
    test_lgb_probs += lgb_model.predict_proba(X_stack_tst) / N_FOLDS
    lgb_f1 = f1_score(y_cls[val_idx], np.argmax(oof_lgb_probs[val_idx], axis=1), average='macro')

    if HAS_CATBOOST:
        cat_model = CatBoostClassifier(
            iterations=1200,
            learning_rate=0.035,
            depth=6,
            l2_leaf_reg=5.0,
            random_seed=42 + fold,
            task_type='CPU',
            verbose=False
        )
        cat_model.fit(
            X_stack_trn, y_cls[trn_idx],
            eval_set=(X_stack_val, y_cls[val_idx]),
            early_stopping_rounds=100,
            verbose=False
        )
        oof_cat_probs[val_idx] = cat_model.predict_proba(X_stack_val)
        test_cat_probs += cat_model.predict_proba(X_stack_tst) / N_FOLDS
        cat_f1 = f1_score(y_cls[val_idx], np.argmax(oof_cat_probs[val_idx], axis=1), average='macro')
    else:
        oof_cat_probs[val_idx] = oof_lgb_probs[val_idx]
        test_cat_probs += test_lgb_probs / N_FOLDS
        cat_f1 = lgb_f1

    print(f"Fold {fold + 1} Summary -> CascadeNN: {best_nn_f1:.4f} | Stacked-LGB: {lgb_f1:.4f} | Stacked-CAT: {cat_f1:.4f}")

# =====================================================================
# 6. OPTIMAL ENSEMBLE BLENDING, THRESHOLD TUNING & SUBMISSION
# =====================================================================
anchor_mask = (is_anchor == 1)
best_ens_f1 = 0.0
best_w = (0.34, 0.33, 0.33)

for w1 in np.linspace(0.15, 0.65, 11):
    for w2 in np.linspace(0.15, 0.65, 11):
        w3 = max(0.0, 1.0 - w1 - w2)
        blend = w1 * oof_nn_probs + w2 * oof_lgb_probs + w3 * oof_cat_probs
        # Optimize blend weights on leak-free non-overlapping anchor windows
        s = f1_score(y_cls[anchor_mask], np.argmax(blend[anchor_mask], axis=1), average='macro')
        if s > best_ens_f1:
            best_ens_f1 = s
            best_w = (w1, w2, w3)

oof_blend = best_w[0] * oof_nn_probs + best_w[1] * oof_lgb_probs + best_w[2] * oof_cat_probs
test_blend = best_w[0] * test_nn_probs + best_w[1] * test_lgb_probs + best_w[2] * test_cat_probs

# Coordinate-Ascent Class Multiplier Tuning on Non-Overlapping Windows
class_mult = np.ones(6, dtype=np.float32)
tuned_anchor_f1 = best_ens_f1
for _ in range(4):
    for c in range(6):
        for m in np.linspace(0.70, 1.40, 36):
            cand = class_mult.copy()
            cand[c] = m
            s = f1_score(
                y_cls[anchor_mask],
                np.argmax(oof_blend[anchor_mask] * cand[None, :], axis=1),
                average='macro'
            )
            if s > tuned_anchor_f1:
                tuned_anchor_f1 = s
                class_mult = cand

full_oof_f1 = f1_score(y_cls, np.argmax(oof_blend * class_mult[None, :], axis=1), average='macro')

print("\n========================================================")
print(f"OOF Macro-F1 (Cascade-NN):      {f1_score(y_cls, np.argmax(oof_nn_probs, axis=1), average='macro'):.4f}")
print(f"OOF Macro-F1 (Stacked-LGBM):    {f1_score(y_cls, np.argmax(oof_lgb_probs, axis=1), average='macro'):.4f}")
print(f"OOF Macro-F1 (Stacked-CatB):    {f1_score(y_cls, np.argmax(oof_cat_probs, axis=1), average='macro'):.4f}")
print(f"OOF Macro-F1 (Anchor Clean):    {tuned_anchor_f1:.4f} | Weights: {np.round(best_w, 2)}")
print(f"OOF Macro-F1 (Full Tuned):      {full_oof_f1:.4f}")
print("========================================================")
print(classification_report(y_cls, np.argmax(oof_blend * class_mult[None, :], axis=1), target_names=GESTURES))

# Generate Final submission.csv
final_preds = np.argmax(test_blend * class_mult[None, :], axis=1)
final_labels = [INV_GESTURE_MAP[i] for i in final_preds]

sample_sub_files = glob.glob(os.path.join(DATA_DIR, "**", "sample_submission*.csv"), recursive=True)
if sample_sub_files:
    sub = pd.read_csv(sample_sub_files[0])
    id_col, target_col = sub.columns[0], sub.columns[1]
    if len(sub) == len(final_labels) and len(set(test_ids)) == len(test_ids):
        pred_map = dict(zip(test_ids, final_labels))
        sub[target_col] = sub[id_col].map(pred_map).fillna(pd.Series(final_labels))
    else:
        sub[target_col] = final_labels
else:
    sub = pd.DataFrame({'action_id': test_ids, 'gesture': final_labels})

sub.to_csv("submission.csv", index=False)
print("\nSaved V3 Kinematic-Cascade submission.csv successfully! Class Counts:")
print(sub.iloc[:, 1].value_counts())